# SHV R60 / LC-Q31-A0 — Consumer-scoped Loss Lifetime Qualification

**目的**：从 R59 的 AdamW8bit＋Transformer Checkpointing 强基准出发，用相同工作负载比较 Full 有效 token logits 与按 256 token 分块、反向重计算的 CrossEntropy。

**研究边界**：这是通用系统机制预检，**不是 SHV 独家创新**。所有 R60 判定仅以本轮 FULL/TILED 同环境对照为准，不直接使用 R59 历史数值作速度评定。先 CPU/GPU 数值验证、再 S128 Pilot，达到门槛才继续 S512。没有 GPU 时导出完整失败包；不解封 Guard、不做长训练。

选择 Colab GPU（Tesla T4 或 A100），从上往下运行。结果 ZIP 和 `.sha256` 在最后显示下载链接。

In [ ]:
import importlib.util, subprocess, sys
need=[x for x in ('transformers','bitsandbytes') if importlib.util.find_spec(x) is None]
if need:
    print('Installing missing runtime dependencies:',need)
    subprocess.run([sys.executable,'-m','pip','install','-q',*need],check=True)
import torch
print('PyTorch:',torch.__version__,'CUDA:',torch.version.cuda,'GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO_CUDA')

In [ ]:
import base64,hashlib,io,json,os,pathlib,zipfile
SOURCE_SHA256='1ed316bf7ea810964ea36967a9cebc914984c1b200c466a518e6794981dff536'
SOURCE_B64=(
    '...'
 )
blob=base64.b64decode(''.join(SOURCE_B64))
assert hashlib.sha256(blob).hexdigest()==SOURCE_SHA256, 'embedded source integrity mismatch'
ROOT=pathlib.Path('/content/shv_r60_lc_q31_a0') if pathlib.Path('/content').exists() else pathlib.Path.cwd()/'shv_r60_lc_q31_a0'
ROOT.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    assert z.testzip() is None,'CRC failure'
    manifest=json.loads(z.read('MODULE_SHA256.json'))
    for name,expected in manifest.items():
        target=(ROOT/name).resolve()
        assert target.is_relative_to(ROOT.resolve()),'unsafe archive path'
        payload=z.read(name)
        assert hashlib.sha256(payload).hexdigest()==expected, name
        target.parent.mkdir(parents=True,exist_ok=True)
        target.write_bytes(payload)
print('R60 source verified:',len(manifest),'files',ROOT)


In [ ]:
import zipfile,json
import os,sys,subprocess,pathlib
ENV=os.environ.copy();ENV['PYTHONPATH']=str(ROOT)+os.pathsep+ENV.get('PYTHONPATH','')
print('CPU contract tests:')
tests=subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],cwd=str(ROOT),env=ENV)
if tests.returncode!=0:raise RuntimeError('CPU tests failed; do not run GPU qualification')
OUT=pathlib.Path.cwd()/'SHV_R60_RESULTS';OUT.mkdir(exist_ok=True)
print('R60 GPU qualification (or NO_CUDA archive):')
experiment=subprocess.run([sys.executable,'-m','shv_r60','--output',str(OUT)],cwd=str(ROOT),env=ENV)
if experiment.returncode:print('Experiment process return code:',experiment.returncode)
result_zip=OUT/'SHV_R60_LC_Q31_A0_RESULT_UPLOAD_TO_CHATGPT.zip'
result_sha=OUT/'SHV_R60_LC_Q31_A0_RESULT_UPLOAD_TO_CHATGPT.zip.sha256'
if not result_zip.exists() or not result_sha.exists():raise RuntimeError('Result files not exported')
with zipfile.ZipFile(result_zip) as z:
    rec=json.loads(z.read('summary.json'))
print('QUALIFICATION STATUS:',rec['status'])
from IPython.display import display,FileLink
display(FileLink(str(result_zip)))
display(FileLink(str(result_sha)))


**回传：** `SHV_R60_LC_Q31_A0_RESULT_UPLOAD_TO_CHATGPT.zip` 与同名 `.zip.sha256`。若状态为 HOLD/FAIL，直接回传原始结果，不要修改 Gate 或删除失败证据。